# Numerical examples for the q-Pascal paper

"Applications of q-Pascal Distribution Series to Certain Subclasses of Analytic Functions"
Y. Aydın, S. Çakmak, S. Yalçın

This notebook computes Table 1 and draws Figures 1–3 of the paper.
Equation, lemma and theorem numbers below refer to the revised version of the paper.

**How to run:** choose *Runtime → Run all*.

Each value in Table 1 is computed in two ways:
* from the closed forms in Lemma 3 (sums $S_0, S_1, S_2$) and Lemma 4 (sums $T_0, T_1, T_2$),
* by the sum of the first 3000 terms of the series.

The two values must agree. The code stops with an error if they differ by more than $10^{-9}$.

## 1. Functions

In [ ]:
import csv
import numpy as np
import matplotlib.pyplot as plt

N_TERMS = 3000  # number of terms in the sums of the series


# ---------- q-notation (Section 1) ----------
def qnum(n, q):
    """q-number [n]_q = (1 - q^n)/(1 - q)."""
    return (1 - q**n) / (1 - q)


def qbinom(n, k, q):
    """q-binomial coefficient."""
    p = 1.0
    for i in range(1, k + 1):
        p *= qnum(n - k + i, q) / qnum(i, q)
    return p


def qpoch(t, q, k):
    """q-Pochhammer symbol (t; q)_k."""
    p = 1.0
    for i in range(k):
        p *= 1 - t * q**i
    return p


def sigma(t, q, k):
    """sigma_k(theta) of (10)."""
    return sum(t * q**i / (1 - t * q**i) for i in range(k))


# ---------- coefficients of the class ----------
def Phi(n, q, alpha, beta, gamma):
    """Phi_n of (3)."""
    return (qnum(n, q) * ((1 - alpha * beta) * (1 + gamma * qnum(n - 1, q))
                          - (1 - beta) * alpha * gamma)
            - alpha * (1 - beta) * (1 - gamma))


def weights(alpha, beta, gamma):
    """Coefficients of [n]_q[n-1]_q, [n]_q and 1 in Phi_n."""
    return ((1 - alpha * beta) * gamma,
            1 - alpha * beta - alpha * gamma * (1 - beta),
            -alpha * (1 - beta) * (1 - gamma))


# ---------- closed forms ----------
def S_sums(t, q, r):
    """S_0, S_1, S_2 of Lemma 3."""
    R, R1 = qnum(r, q), qnum(r + 1, q)
    S0 = 1 / qpoch(t, q, r) - 1
    S1 = R * t / qpoch(t, q, r + 1) + 1 / qpoch(q * t, q, r) - 1
    S2 = R * R1 * t**2 / qpoch(t, q, r + 2) + (1 + q) * R * t / qpoch(q * t, q, r + 1)
    return S0, S1, S2


def T_sums(t, q, r):
    """T_0, T_1, T_2 of Lemma 4."""
    R, R1 = qnum(r, q), qnum(r + 1, q)
    T0 = (1 + sigma(t, q, r)) / qpoch(t, q, r) - 1
    T1 = (R * t * (2 + sigma(t, q, r + 1)) / qpoch(t, q, r + 1)
          + (1 + sigma(q * t, q, r)) / qpoch(q * t, q, r) - 1)
    T2 = (R * R1 * t**2 * (3 + sigma(t, q, r + 2)) / qpoch(t, q, r + 2)
          + (1 + q) * R * t * (2 + sigma(q * t, q, r + 1)) / qpoch(q * t, q, r + 1))
    return T0, T1, T2


def G(t, q, r, alpha, beta, gamma):
    """Left-hand side of (8): condition of Theorems 1 and 2."""
    w2, w1, w0 = weights(alpha, beta, gamma)
    S0, S1, S2 = S_sums(t, q, r)
    return qpoch(t, q, r) * (w2 * S2 + w1 * S1 + w0 * S0)


def H(t, q, r, alpha, beta, gamma):
    """Left-hand side of (12): condition of Theorem 3."""
    w2, w1, w0 = weights(alpha, beta, gamma)
    T0, T1, T2 = T_sums(t, q, r)
    return qpoch(t, q, r) * (w2 * T2 + w1 * T1 + w0 * T0)


# ---------- sums of the series ----------
def series_sum(t, q, r, alpha, beta, gamma, weight=lambda n: 1.0):
    """(theta;q)_r * sum_{n>=2} weight(n) Phi_n [n+r-2 choose r-1]_q theta^(n-1).
    weight = 1 gives G, weight = n gives H."""
    return qpoch(t, q, r) * sum(
        weight(n) * Phi(n, q, alpha, beta, gamma)
        * qbinom(n + r - 2, r - 1, q) * t**(n - 1)
        for n in range(2, N_TERMS))


# ---------- theta* ----------
def theta_star(F, q, r, alpha, beta, gamma):
    """theta in [0,1) with F(theta) = 1 - alpha (bisection)."""
    lo, hi = 0.0, 1.0 - 1e-12
    for _ in range(100):
        mid = (lo + hi) / 2
        if F(mid, q, r, alpha, beta, gamma) <= 1 - alpha:
            lo = mid
        else:
            hi = mid
    return lo


def is_increasing(F, q, r, alpha, beta, gamma, upper=0.95, points=400):
    """Check that F increases with theta on [0, upper]."""
    v = [F(upper * j / points, q, r, alpha, beta, gamma) for j in range(points + 1)]
    return all(v[j + 1] > v[j] for j in range(points))


# ---------- the quotient w = N(z)/D(z) of Definition 1 ----------
def series_coefficients(t, q, r, n_max=150):
    """Coefficients a_0, ..., a_{n_max-1} of P_q^{theta,r} in (6)."""
    a = np.zeros(n_max)
    a[1] = 1.0
    P = qpoch(t, q, r)
    for n in range(2, n_max):
        a[n] = qbinom(n + r - 2, r - 1, q) * t**(n - 1) * P
    return a


def quotient(z, q, alpha, beta, gamma, a):
    """w = N(z)/D(z), with A_n and B_n as in the proof of Lemma 1."""
    n = np.arange(len(a))
    qn = np.array([qnum(k, q) for k in n])
    qn1 = np.array([qnum(k - 1, q) if k >= 1 else 0.0 for k in n])
    A = qn * (1 + gamma * qn1)
    B = gamma * qn * (1 + beta * qn1) + (1 - gamma) * (beta * qn + 1 - beta)
    powers = np.power.outer(z, n)
    return (powers @ (A * a)) / (powers @ (B * a))


# ---------- Table 1 ----------
TABLE_ROWS = [
    # (row, result, q, r, alpha, beta, gamma, theta)
    (1, "Thm 1, 2", 0.7, 2, 0.20, 0.10, 0.5, 0.15),
    (2, "Thm 1, 2", 0.7, 2, 0.20, 0.10, 0.5, 0.25),
    (3, "Cor 3",    0.5, 3, 0.10, 0.05, 0.0, 0.40),
    (4, "Cor 3",    0.5, 3, 0.10, 0.05, 0.0, 0.45),
    (5, "Cor 4",    0.8, 4, 0.15, 0.10, 1.0, 0.05),
    (6, "Cor 4",    0.8, 4, 0.15, 0.10, 1.0, 0.10),
    (7, "Thm 3",    0.6, 2, 0.25, 0.10, 0.8, 0.03),
    (8, "Thm 3",    0.6, 2, 0.25, 0.10, 0.8, 0.10),
]


def make_table(filename="table1.csv"):
    header = ["row", "result", "q", "r", "alpha", "beta", "gamma", "theta",
              "theta_star", "LHS_closed_form", "LHS_sum", "1-alpha", "holds"]
    rows = []
    for row, res, q, r, a, b, g, t in TABLE_ROWS:
        if res == "Thm 3":   # condition (12)
            F, weight = H, (lambda n: n)
        else:                # condition (8)
            F, weight = G, (lambda n: 1.0)
        closed = F(t, q, r, a, b, g)
        summed = series_sum(t, q, r, a, b, g, weight)
        assert abs(closed - summed) < 1e-9, f"row {row}: closed form and sum differ"
        assert is_increasing(F, q, r, a, b, g), f"row {row}: LHS is not increasing"
        ts = theta_star(F, q, r, a, b, g)
        rows.append([row, res, q, r, a, b, g, t, round(ts, 3), round(closed, 3),
                     round(summed, 3), round(1 - a, 2), "Yes" if closed <= 1 - a else "No"])
    with open(filename, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)
    return header, rows


# ---------- Figures ----------
plt.rcParams.update({"font.size": 13, "axes.labelsize": 15, "axes.titlesize": 15,
                     "legend.fontsize": 12, "font.family": "serif",
                     "mathtext.fontset": "cm"})
RADII = [0.5, 0.75, 0.9, 0.999]
RADIUS_COLORS = ["#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]
Q_COLORS = {0.3: "#2a78d6", 0.6: "#eb6834", 0.9: "#1baf7a"}
LINE_COLOR = "#eb6834"   # line Re w = alpha
SHADE_COLOR = "#f0efec"  # half-plane Re w < alpha


def draw_quotient(ax, q, alpha, beta, gamma, a, title=None, legend="upper right"):
    ax.axvspan(-1, alpha, color=SHADE_COLOR, zorder=0)
    ax.axvline(alpha, color=LINE_COLOR, ls="--", lw=1.5, zorder=1)
    ax.text(alpha + 0.02, -0.42, r"$\mathrm{Re}\,w=\alpha$", fontsize=12,
            rotation=90, va="bottom")
    for rho, col in zip(RADII, RADIUS_COLORS):
        z = rho * np.exp(1j * np.linspace(0, 2 * np.pi, 2000))
        w = quotient(z, q, alpha, beta, gamma, a)
        ax.plot(w.real, w.imag, color=col, lw=2, label=rf"$\rho={rho}$", zorder=2)
    if title:
        ax.set_title(title)
    if legend:
        ax.legend(loc=legend, fontsize=11, frameon=False)
    ax.set_xlabel(r"$\mathrm{Re}\,w$")
    ax.set_ylabel(r"$\mathrm{Im}\,w$")
    ax.set_aspect("equal")
    ax.set_xlim(0, 1.6)
    ax.set_ylim(-0.5, 0.5)
    ax.grid(color="#dddddd", lw=0.6, zorder=0)
    z = 0.999 * np.exp(1j * np.linspace(0, 2 * np.pi, 4000))
    return quotient(z, q, alpha, beta, gamma, a).real.min()


def make_figures():
    # Figure 1: G(theta) of (8) for three values of q
    r, a, b, g = 2, 0.2, 0.1, 0.5
    fig, ax = plt.subplots(figsize=(6.4, 4.4))
    th = np.linspace(0, 0.6, 400)
    for q, ls in [(0.3, "-"), (0.6, "--"), (0.9, "-.")]:
        ax.plot(th, [G(t, q, r, a, b, g) for t in th], ls, color=Q_COLORS[q],
                lw=2.2, label=f"$q={q}$")
        ts = theta_star(G, q, r, a, b, g)
        ax.plot([ts], [1 - a], "o", color=Q_COLORS[q], ms=8,
                markeredgecolor="white", markeredgewidth=1.5, zorder=3)
        ax.annotate(f"{ts:.3f}", (ts, 1 - a), textcoords="offset points",
                    xytext=(4, -16), fontsize=11)
        print(f"Figure 1: q = {q}, theta* = {ts:.3f}")
    ax.axhline(1 - a, color="0.35", lw=1.2)
    ax.grid(color="#dddddd", lw=0.6)
    ax.text(0.005, 1 - a + 0.04, r"$1-\alpha$", fontsize=13)
    ax.set_xlabel(r"$\theta$")
    ax.set_ylabel(r"$G(\theta)$")
    ax.set_xlim(0, 0.6)
    ax.set_ylim(0, 1.6)
    ax.legend(loc="upper left", bbox_to_anchor=(0.02, 0.88))
    fig.tight_layout()
    fig.savefig("fig1.pdf")

    # Figure 2: w = N/D for P and TP, parameters of row 1 of Table 1
    q, r, t, a, b, g = 0.7, 2, 0.15, 0.2, 0.1, 0.5
    c = series_coefficients(t, q, r)
    cT = c.copy()
    cT[2:] = -cT[2:]   # TP of (7)
    fig, axs = plt.subplots(1, 2, figsize=(11.2, 3.9))
    m1 = draw_quotient(axs[0], q, a, b, g, c, r"$f=P_q^{\theta,r}$", legend=None)
    m2 = draw_quotient(axs[1], q, a, b, g, cT, r"$f=\mathcal{T}P_q^{\theta,r}$", legend=None)
    print(f"Figure 2: smallest Re w on |z|=0.999: P {m1:.3f}, TP {m2:.3f}")
    handles, labels = axs[1].get_legend_handles_labels()
    fig.tight_layout(rect=(0, 0, 0.88, 1))
    fig.legend(handles, labels, loc="center right", fontsize=11, frameon=False)
    fig.savefig("fig2.pdf")

    # Figure 3: w = N/D for P * k, k the Koebe function, parameters of row 7
    q, r, t, a, b, g = 0.6, 2, 0.03, 0.25, 0.1, 0.8
    c = series_coefficients(t, q, r)
    c = c * np.arange(len(c))  # Koebe function: a_n = n
    fig, ax = plt.subplots(figsize=(5.4, 3.9))
    m3 = draw_quotient(ax, q, a, b, g, c)
    print(f"Figure 3: smallest Re w on |z|=0.999: {m3:.3f}")
    fig.tight_layout()
    fig.savefig("fig3.pdf")


## 2. Table 1

Rows 1–6 use condition (8) (Theorems 1–2 and Corollaries 3–4). Rows 7–8 use condition (12) (Theorem 3).
For each row, the code also checks that the left-hand side increases with θ on [0, 0.95]. So θ* is the only value where equality holds.

In [ ]:
import pandas as pd
make_table()
pd.read_csv("table1.csv")

## 3. Figures 1–3

The figures are shown below. They are also saved as `fig1.pdf`, `fig2.pdf` and `fig3.pdf` in the file panel on the left.

In [ ]:
make_figures()
plt.show()

## 4. Download the files (optional, Google Colab only)

In [ ]:
from google.colab import files
for name in ["table1.csv", "fig1.pdf", "fig2.pdf", "fig3.pdf"]:
    files.download(name)